## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("✓ Imports complete")

✓ Imports complete


## Configuration

In [2]:
# import os
# import gc
# from tqdm import tqdm

# START_DATE = "2025-06-01"
# END_DATE = "2025-06-01"

# DATA_DIR = "/home/ubuntu/data/uploads/objects/clean"
# CHUNK_SIZE = 500000

# def load_data(data_dir, start_date, end_date):
#     """
#     Load data with optimized dtypes for memory efficiency
#     """
#     # Define optimal dtypes upfront - saves ~50% memory
#     dtypes = {
#         'id': 'int32',           # Was int64 → Save 50%
#         'label': 'int8',         # Values 1-8 → Save 87.5%
#         'pos_x': 'float32',      # Was float64 → Save 50%
#         'pos_y': 'float32',
#         'pos_z': 'float32',
#         'vel': 'float32',
#         'vel_x': 'float32',
#         'vel_y': 'float32',
#         'yaw': 'float32',
#         'size_x': 'float32',
#         'size_y': 'float32',
#         # timestamp stays int64 (needed for precision)
#     }
    
#     # List to collect dataframes
#     dfs = []
    
#     # Loop over all folders within date range
#     for folder in tqdm(sorted(os.listdir(data_dir)), desc="Loading data"):
#         folder_path = os.path.join(data_dir, folder)
        
#         # Skip non-folders
#         if not os.path.isdir(folder_path):
#             continue
        
#         if folder.startswith(start_date) or folder.startswith(end_date):
#             # Load all parquet files inside the folder
#             df_chunk = pd.read_parquet(folder_path)
            
#             # Apply dtypes to matching columns
#             for col, dtype in dtypes.items():
#                 if col in df_chunk.columns:
#                     df_chunk[col] = df_chunk[col].astype(dtype)
            
#             dfs.append(df_chunk)
#             del df_chunk  # Clean up immediately
    
#     # Combine all into single dataframe
#     if dfs:
#         df = pd.concat(dfs, ignore_index=True)
#         # Destroy the list immediately
#         del dfs
#         gc.collect()
#         return df
#     else:
#         print("No data found for given date range.")
#         return pd.DataFrame()

# # Usage
# df = load_data(DATA_DIR, START_DATE, END_DATE)
# print(f"Loaded {len(df)} records from {START_DATE} to {END_DATE}")

In [17]:
# Input paths
MDRAC_INPUT = 'results/mdrac/04/mdrac_04.csv'
SPF_INPUT = 'results/spf/spf_conflicts.csv'

# Output paths
MDRAC_OUTPUT = 'results/mdrac/06/mdrac_06_postprocessed.csv'
SPF_OUTPUT = 'results/spf/spf_conflicts_processed.csv'

# Filter thresholds
MAX_YAW_DIFF = 170.0  # degrees - exclude near head-on conflicts

print(f"Configuration:")
print(f"  Max yaw difference: {MAX_YAW_DIFF}°")
print(f"  M-DRAC input: {MDRAC_INPUT}")
print(f"  SPF input: {SPF_INPUT}")

Configuration:
  Max yaw difference: 170.0°
  M-DRAC input: results/mdrac/04/mdrac_04.csv
  SPF input: results/spf/spf_conflicts.csv


---
## Utility Functions

In [11]:
def create_pair_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create normalized pair_id (always smaller ID first).
    Ensures (id1=100, id2=200) and (id1=200, id2=100) are treated as same pair.
    """
    df = df.copy()
    
    id_min = np.minimum(df['id1'].values, df['id2'].values)
    id_max = np.maximum(df['id1'].values, df['id2'].values)
    
    df['pair_id'] = id_min.astype(str) + '_' + id_max.astype(str)
    
    return df


def apply_yaw_filter(df: pd.DataFrame, max_yaw_diff: float) -> pd.DataFrame:
    """
    Filter conflicts by yaw difference threshold.
    Removes near head-on conflicts (opposite directions).
    """
    initial_count = len(df)
    filtered = df[df['yaw_diff'] < max_yaw_diff].copy()
    removed = initial_count - len(filtered)
    
    print(f"  Yaw filter (< {max_yaw_diff}°): {removed:,} removed ({removed/initial_count*100:.1f}%)")
    return filtered


def deduplicate_pairs(df: pd.DataFrame, metric_col: str) -> pd.DataFrame:
    """
    Group by pair_id and select worst case.
    
    Args:
        df: DataFrame with pair_id column
        metric_col: Column to maximize ('MDRAC' or 'composite_risk')
    
    Returns:
        Deduplicated DataFrame (one row per pair)
    """
    initial_count = len(df)
    unique_pairs = df['pair_id'].nunique()
    
    # Sort by metric (desc) and distance (asc) for tiebreaker
    df_sorted = df.sort_values([metric_col, 'dist'], ascending=[False, True])
    
    # Keep first row per pair (worst case)
    deduplicated = df_sorted.groupby('pair_id', as_index=False).first()
    
    reduction = (1 - len(deduplicated) / initial_count) * 100
    print(f"  Deduplication: {initial_count:,} → {len(deduplicated):,} ({reduction:.1f}% reduction)")
    print(f"  Avg detections per pair: {initial_count/unique_pairs:.1f}")
    
    return deduplicated


---
# M-DRAC Post-Processing

## Load M-DRAC Detections

In [12]:
print("="*70)
print("M-DRAC POST-PROCESSING")
print("="*70)

mdrac_raw = pd.read_csv(MDRAC_INPUT)
print(f"\nLoaded {len(mdrac_raw):,} raw M-DRAC detections")
print(f"Columns: {list(mdrac_raw.columns)}")
print(f"\nSample:")
mdrac_raw.head(3)

M-DRAC POST-PROCESSING

Loaded 10 raw M-DRAC detections
Columns: ['timestamp', 'id1', 'id2', 'zone', 'interaction', 'leader', 'dist', 'TTC', 'MDRAC', 'closing_speed', 'speed_diff', 'yaw_diff', 'link']

Sample:


,timestamp,id1,id2,zone,interaction,leader,dist,TTC,MDRAC,closing_speed,speed_diff,yaw_diff,link
0,2025-06-04 04:15:37.974562883,13130564,13130814,A-L2,van_v_car,13130564,6.403067,0.911280,4.118060,7.026452,0.994987,3.580986,https://di-india-collab-2.flow-analytics.io/to...
1,2025-06-04 05:13:55.341478109,13168548,13168566,E-L2,car_v_car,13168548,6.246952,0.788952,4.966669,7.918040,10.488945,2.573834,https://di-india-collab-2.flow-analytics.io/to...
2,2025-06-04 08:42:05.250216007,13358849,13359199,D-L2,bus_v_car,13359199,5.268777,0.590091,6.489853,8.928752,8.347487,16.002531,https://di-india-collab-2.flow-analytics.io/to...


## Apply Filters

In [14]:
print(f"\nApplying filters...")

# Step 1: Yaw filter
mdrac_filtered = apply_yaw_filter(mdrac_raw, MAX_YAW_DIFF)

# Step 2: Create pair IDs
mdrac_filtered = create_pair_id(mdrac_filtered)

# Step 3: Deduplicate by pair (keep worst MDRAC)
mdrac_processed = deduplicate_pairs(mdrac_filtered, metric_col='MDRAC')

# Remove temporary pair_id column
mdrac_output = mdrac_processed.drop(columns=['pair_id'])

print(f"\n✓ M-DRAC processing complete")
print(f"  Final conflicts: {len(mdrac_output):,}")
print(f"  Total reduction: {(1 - len(mdrac_output)/len(mdrac_raw))*100:.1f}%")


Applying filters...
  Yaw filter (< 170.0°): 0 removed (0.0%)
  Deduplication: 10 → 10 (0.0% reduction)
  Avg detections per pair: 1.0

✓ M-DRAC processing complete
  Final conflicts: 10
  Total reduction: 0.0%


## M-DRAC Analysis

In [15]:
print("\n" + "="*70)
print("M-DRAC ANALYSIS")
print("="*70)

print("\nMDRAC Statistics:")
print(mdrac_output['MDRAC'].describe())

print("\nInteraction Type Distribution:")
print(mdrac_output['interaction'].value_counts())

print("\nTop 10 Highest MDRAC Conflicts:")
top10_mdrac = mdrac_output.nlargest(10, 'MDRAC')[[
    'timestamp', 'id1', 'id2', 'interaction', 'leader', 
    'dist', 'TTC', 'MDRAC', 'yaw_diff'
]]
print(top10_mdrac.to_string(index=False))


M-DRAC ANALYSIS

MDRAC Statistics:
count    10.000000
mean      7.912324
std       7.058656
min       3.919024
25%       4.130785
50%       5.244472
75%       6.282882
max      26.012527
Name: MDRAC, dtype: float64

Interaction Type Distribution:
interaction
car_v_car    6
van_v_car    2
bus_v_car    1
car_v_van    1
Name: count, dtype: int64

Top 10 Highest MDRAC Conflicts:
                    timestamp      id1      id2 interaction   leader     dist      TTC     MDRAC  yaw_diff
2025-06-04 22:46:05.458056927 14030238 14030253   car_v_car 14030238 7.923012 0.692245 26.012527  4.252421
2025-06-04 16:54:24.591604948 13864660 13864975   van_v_car 13864660 7.019087 0.773199 14.230817  1.454776
2025-06-04 08:42:05.250216007 13358849 13359199   bus_v_car 13359199 5.268777 0.590091  6.489853 16.002531
2025-06-04 11:53:13.493012905 13550913 13550957   car_v_car 13550913 6.793139 0.771389  5.661967  0.111906
2025-06-04 13:33:29.194168091 13644543 13645439   car_v_car 13644543 6.746762 0.809436

## Save M-DRAC Results

In [18]:
mdrac_output.to_csv(MDRAC_OUTPUT, index=False)
print(f"\n✓ Saved M-DRAC processed conflicts to: {MDRAC_OUTPUT}")
print(f"  Rows: {len(mdrac_output):,}")


✓ Saved M-DRAC processed conflicts to: results/mdrac/06/mdrac_06_postprocessed.csv
  Rows: 10


---
# SPF Post-Processing

## Load SPF Detections

In [ ]:
print("\n" + "="*70)
print("SPF POST-PROCESSING")
print("="*70)

spf_raw = pd.read_csv(SPF_INPUT)
print(f"\nLoaded {len(spf_raw):,} raw SPF detections")
print(f"Columns: {list(spf_raw.columns)}")
print(f"\nSample:")
spf_raw.head(3)


SPF POST-PROCESSING

Loaded 3,298 raw SPF detections
Columns: ['timestamp', 'id1', 'id2', 'interaction', 'dist', 'TTC', 'composite_risk', 'closing_speed', 'speed_diff', 'yaw_diff', 'link']

Sample:


,timestamp,id1,id2,interaction,dist,TTC,composite_risk,closing_speed,speed_diff,yaw_diff,link
0,2025-06-05 03:23:12.939105988,14067457,14067485,car_v_car,7.983550,5.825747,0.593790,1.370391,1.562821,1.902399,https://di-india-collab.flow-analytics.io/tool...
1,2025-06-05 03:23:13.035861015,14067457,14067485,car_v_car,7.893116,6.258413,0.549583,1.261201,1.523839,1.566681,https://di-india-collab.flow-analytics.io/tool...
2,2025-06-05 03:28:03.539572001,14069271,14069277,van_v_car,7.888938,3.005890,0.909780,2.624494,0.504483,0.111906,https://di-india-collab.flow-analytics.io/tool...


In [ ]:
# =============================================================================
# FILTER: Teleportation Timestamps
# =============================================================================
# Remove conflicts at timestamps where vehicles have unrealistic position jumps

from filters.teleportation_filter import filter_conflicts_by_teleportation, MAX_JUMP_DISTANCE, SAMPLING_RATE

# Use same vehicle data from M-DRAC section
if 'df' not in locals():
    print("⚠️  Skipping teleportation filter (df not loaded)")
else:
    print(f"Original vehicle data: {len(df):,} records")
    spf_raw = filter_conflicts_by_teleportation(
        conflicts=spf_raw,
        vehicle_data=df,
        max_jump=MAX_JUMP_DISTANCE,
        sampling_rate=SAMPLING_RATE,
        verbose=True
    )

⚠️  Skipping teleportation filter (df not loaded)


## Apply Filters

In [ ]:
print(f"\nApplying filters...")

# Step 1: Yaw filter
spf_filtered = apply_yaw_filter(spf_raw, MAX_YAW_DIFF)

# Step 2: Create pair IDs
spf_filtered = create_pair_id(spf_filtered)

# Step 3: Deduplicate by pair (keep highest composite_risk)
spf_processed = deduplicate_pairs(spf_filtered, metric_col='composite_risk')

# Remove temporary pair_id column
spf_output = spf_processed.drop(columns=['pair_id'])

print(f"\n✓ SPF processing complete")
print(f"  Final conflicts: {len(spf_output):,}")
print(f"  Total reduction: {(1 - len(spf_output)/len(spf_raw))*100:.1f}%")


Applying filters...
  Yaw filter (< 170.0°): 25 removed (0.8%)
  Deduplication: 3,273 → 509 (84.4% reduction)
  Avg detections per pair: 6.4

✓ SPF processing complete
  Final conflicts: 509
  Total reduction: 84.6%


## SPF Analysis

In [ ]:
print("\n" + "="*70)
print("SPF ANALYSIS")
print("="*70)

print("\nComposite Risk Statistics:")
print(spf_output['composite_risk'].describe())

print("\nInteraction Type Distribution:")
print(spf_output['interaction'].value_counts())

# print("\nTop 10 Highest Risk Conflicts:")
# top10_spf = spf_output.nlargest(10, 'composite_risk')[[
#     'timestamp', 'id1', 'id2', 'interaction', 
#     'dist', 'TTC', 'composite_risk', 'yaw_diff'
# ]]
# print(top10_spf.to_string(index=False))


SPF ANALYSIS

Composite Risk Statistics:
count    509.000000
mean       0.715947
std        0.149585
min        0.500671
25%        0.589587
50%        0.688720
75%        0.826838
max        0.999396
Name: composite_risk, dtype: float64

Interaction Type Distribution:
interaction
car_v_car      429
van_v_car       28
car_v_van       25
truck_v_car      8
van_v_van        6
bus_v_car        4
car_v_truck      4
car_v_bus        4
truck_v_van      1
Name: count, dtype: int64


## Save SPF Results

In [ ]:
spf_output.to_csv(SPF_OUTPUT, index=False)
print(f"\n✓ Saved SPF processed conflicts to: {SPF_OUTPUT}")
print(f"  Rows: {len(spf_output):,}")


✓ Saved SPF processed conflicts to: results/postprocessed_conflicts/spf_conflicts_processed.csv
  Rows: 509


---
# Summary Statistics

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

summary = pd.DataFrame({
    'Method': ['M-DRAC', 'SPF'],
    'Raw Detections': [len(mdrac_raw), len(spf_raw)],
    'After Yaw Filter': [len(mdrac_filtered), len(spf_filtered)],
    'After Deduplication': [len(mdrac_output), len(spf_output)],
    'Total Reduction %': [
        f"{(1 - len(mdrac_output)/len(mdrac_raw))*100:.1f}%",
        f"{(1 - len(spf_output)/len(spf_raw))*100:.1f}%"
    ]
})

print("\n", summary.to_string(index=False))


FINAL SUMMARY

 Method  Raw Detections  After Yaw Filter  After Deduplication Total Reduction %
M-DRAC               3                 3                    3              0.0%
   SPF            3298              3273                  509             84.6%
